# FINAL_STRONG_SOLUTION

Усиленный ноутбук (на базе идей BEST_SOLUTION): **NoML + ALS + ALS-neighbors aggregates + CatBoostRanker** с защитой от ошибки `All train targets are equal`.

In [ ]:
import os
import math
import random
import warnings
import importlib.util
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from catboost import CatBoostRanker, Pool

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)


In [ ]:
DATA_DIR_CANDIDATES = [
    '/kaggle/input/datasets/artemnazemtsev/nto-ai',
    './data',
]
TOP_K = 20
CANDS_PER_USER = 650
NEG_PER_USER = 260

CATBOOST_PARAMS = {
    'loss_function': 'YetiRank',
    'eval_metric': 'NDCG:top=20',
    'learning_rate': 0.04,
    'depth': 8,
    'l2_leaf_reg': 10.0,
    'min_data_in_leaf': 30,
    'iterations': 1500,
    'random_seed': 42,
    'verbose': 200,
}

for p in DATA_DIR_CANDIDATES:
    if os.path.exists(p):
        DATA_DIR = p
        break
print('DATA_DIR =', DATA_DIR)


In [ ]:
interactions = pd.read_csv(
    f'{DATA_DIR}/interactions.csv',
    parse_dates=['event_ts'],
    dtype={'user_id':'int64', 'edition_id':'int64', 'event_type':'int8'}
)
targets = pd.read_csv(f'{DATA_DIR}/targets.csv', dtype={'user_id':'int64'})
editions = pd.read_csv(f'{DATA_DIR}/editions.csv')
book_genres = pd.read_csv(f'{DATA_DIR}/book_genres.csv')

for c in ['edition_id','book_id','author_id','language_id','publisher_id','publication_year','age_restriction']:
    if c in editions.columns:
        editions[c] = editions[c].fillna(-1)

for c in ['edition_id','book_id','author_id','language_id','publisher_id']:
    if c in editions.columns:
        editions[c] = editions[c].astype('int64')
book_genres['book_id'] = book_genres['book_id'].astype('int64')
book_genres['genre_id'] = book_genres['genre_id'].astype('int64')

pos = interactions[interactions['event_type'].isin([1,2])].copy()
pos = pos.sort_values(['user_id','event_ts'])
max_ts = pos['event_ts'].max()

print('interactions:', interactions.shape)
print('targets:', targets.shape)
print('editions:', editions.shape)
print('date range:', pos['event_ts'].min(), '->', max_ts)


In [ ]:
# Robust pseudo-validation split (чтобы не было all targets equal)
def make_split(df):
    tmax = df['event_ts'].max()
    train_end = tmax - pd.Timedelta(days=21)
    valid_start = tmax - pd.Timedelta(days=49)

    valid = df[(df['event_ts'] >= valid_start) & (df['event_ts'] < train_end)][['user_id','edition_id']].drop_duplicates()
    if len(valid) < 500:
        q1 = df['event_ts'].quantile(0.78)
        q2 = df['event_ts'].quantile(0.92)
        valid = df[(df['event_ts'] >= q1) & (df['event_ts'] < q2)][['user_id','edition_id']].drop_duplicates()
        train_end = q1

    hist = df[df['event_ts'] < train_end].copy()
    return hist, valid

hist_pos, valid_truth = make_split(pos)
valid_truth['label'] = 1
print('hist_pos:', hist_pos.shape, 'valid_truth_pairs:', valid_truth.shape)


In [ ]:
book_to_genres = defaultdict(list)
for b, g in book_genres[['book_id','genre_id']].drop_duplicates().itertuples(index=False):
    book_to_genres[int(b)].append(int(g))

user_pos_full = pos.groupby('user_id')['edition_id'].agg(list).to_dict()
user_pos_hist = hist_pos.groupby('user_id')['edition_id'].agg(list).to_dict()

# popularity priors
pop_all = hist_pos.groupby('edition_id')['user_id'].nunique().rename('pop_all')
recent_cut = hist_pos['event_ts'].max() - pd.Timedelta(days=30)
pop_recent = hist_pos[hist_pos['event_ts'] >= recent_cut].groupby('edition_id')['user_id'].nunique().rename('pop_recent')
item_pop_df = pd.concat([pop_all, pop_recent], axis=1).fillna(0).reset_index()
item_pop_df['pop_all_log'] = np.log1p(item_pop_df['pop_all'])
item_pop_df['pop_recent_log'] = np.log1p(item_pop_df['pop_recent'])

global_top = item_pop_df.sort_values(['pop_all','edition_id'], ascending=[False,True])
recent_top = item_pop_df.sort_values(['pop_recent','edition_id'], ascending=[False,True])


In [ ]:
# User metadata affinity profiles on history
hist_meta = hist_pos[['user_id','edition_id']].merge(
    editions[['edition_id','author_id','language_id','publisher_id','book_id']],
    on='edition_id', how='left'
)

u_author = hist_meta.groupby(['user_id','author_id'], as_index=False).size().rename(columns={'size':'cnt'})
u_lang = hist_meta.groupby(['user_id','language_id'], as_index=False).size().rename(columns={'size':'cnt'})
u_pub = hist_meta.groupby(['user_id','publisher_id'], as_index=False).size().rename(columns={'size':'cnt'})
for df in [u_author, u_lang, u_pub]:
    df['w'] = df['cnt'] / df.groupby('user_id')['cnt'].transform('sum')

grows = []
for u, b in hist_meta[['user_id','book_id']].itertuples(index=False):
    for g in book_to_genres.get(int(b), []):
        grows.append((int(u), int(g)))
ugen = pd.DataFrame(grows, columns=['user_id','genre_id']) if grows else pd.DataFrame(columns=['user_id','genre_id'])
if len(ugen):
    u_genre = ugen.groupby(['user_id','genre_id'], as_index=False).size().rename(columns={'size':'cnt'})
    u_genre['w'] = u_genre['cnt'] / u_genre.groupby('user_id')['cnt'].transform('sum')
else:
    u_genre = pd.DataFrame(columns=['user_id','genre_id','cnt','w'])

# entity -> top editions by popularity

def topk(frame, key_col, item_col='edition_id', score_col='pop_all', k=130):
    d = {}
    for key, g in frame.groupby(key_col):
        gg = g.sort_values([score_col,item_col], ascending=[False,True]).head(k)
        d[int(key)] = list(zip(gg[item_col].astype(int).tolist(), gg[score_col].astype(float).tolist()))
    return d

base = editions[['edition_id','author_id','language_id','publisher_id','book_id']].merge(item_pop_df[['edition_id','pop_all']], on='edition_id', how='left').fillna({'pop_all':0.0})
author2eds = topk(base, 'author_id', score_col='pop_all', k=140)
lang2eds = topk(base, 'language_id', score_col='pop_all', k=140)
pub2eds = topk(base, 'publisher_id', score_col='pop_all', k=140)

gtmp = editions[['edition_id','book_id']].merge(book_genres, on='book_id', how='left').merge(item_pop_df[['edition_id','pop_all']], on='edition_id', how='left').fillna({'pop_all':0.0})
genre2eds = topk(gtmp, 'genre_id', score_col='pop_all', k=140)

ua = u_author.groupby('user_id').apply(lambda g: list(zip(g['author_id'].astype(int), g['w'].astype(float)))).to_dict() if len(u_author) else {}
ul = u_lang.groupby('user_id').apply(lambda g: list(zip(g['language_id'].astype(int), g['w'].astype(float)))).to_dict() if len(u_lang) else {}
up = u_pub.groupby('user_id').apply(lambda g: list(zip(g['publisher_id'].astype(int), g['w'].astype(float)))).to_dict() if len(u_pub) else {}
ug = u_genre.groupby('user_id').apply(lambda g: list(zip(g['genre_id'].astype(int), g['w'].astype(float)))).to_dict() if len(u_genre) else {}


In [ ]:
# Co-visitation from history
user_hist = hist_pos.sort_values('event_ts').groupby('user_id')['edition_id'].agg(list)
co_counts = defaultdict(Counter)
for _, items in tqdm(user_hist.items(), total=len(user_hist), desc='co-vis'):
    uniq, seen = [], set()
    for x in items[-90:]:
        x = int(x)
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    n = len(uniq)
    for i in range(n):
        a = uniq[i]
        for j in range(max(0, i-14), min(n, i+15)):
            if i == j:
                continue
            b = uniq[j]
            co_counts[a][b] += 1

item_pop = hist_pos.groupby('edition_id')['user_id'].nunique().to_dict()
item2top = {}
for a, ctr in tqdm(co_counts.items(), desc='co-vis norm'):
    pa = item_pop.get(a, 1)
    arr = []
    for b, c in ctr.items():
        pb = item_pop.get(b, 1)
        arr.append((b, c / math.sqrt(pa * pb)))
    arr.sort(key=lambda x: (-x[1], x[0]))
    item2top[a] = arr[:140]


In [ ]:
# ALS + ALS neighbor aggregates
HAS_IMPLICIT = importlib.util.find_spec('implicit') is not None
als_recs = {}
als_simitem_agg = defaultdict(dict)
als_userknn_agg = defaultdict(dict)

if HAS_IMPLICIT:
    import implicit
    from scipy.sparse import csr_matrix

    tmp = hist_pos[['user_id','edition_id','event_type']].copy()
    tmp['w'] = np.where(tmp['event_type'] == 2, 1.7, 1.1).astype('float32')

    uids = np.sort(tmp['user_id'].unique())
    iids = np.sort(tmp['edition_id'].unique())
    u2i = {u:i for i,u in enumerate(uids)}
    it2i = {it:i for i,it in enumerate(iids)}
    i2it = {i:it for it,i in it2i.items()}

    rows = tmp['user_id'].map(u2i).to_numpy()
    cols = tmp['edition_id'].map(it2i).to_numpy()
    vals = tmp['w'].to_numpy()
    mat = csr_matrix((vals, (rows, cols)), shape=(len(uids), len(iids)))

    als = implicit.als.AlternatingLeastSquares(
        factors=128,
        regularization=0.03,
        iterations=30,
        random_state=42,
        use_gpu=False,
    )
    als.fit(mat)

    # 1) direct ALS recommend
    for u in tqdm(targets['user_id'].tolist(), desc='als recs'):
        if u not in u2i:
            continue
        uid = u2i[u]
        rec_idx, rec_sc = als.recommend(uid, mat[uid], N=200, filter_already_liked_items=False)
        als_recs[int(u)] = [(int(i2it[int(ii)]), float(sc)) for ii, sc in zip(rec_idx, rec_sc)]

    # 2) aggregate by similar items to last user items
    for u in tqdm(targets['user_id'].tolist(), desc='als similar_items agg'):
        if u not in user_pos_hist:
            continue
        agg = defaultdict(float)
        seq = user_pos_hist[u][-35:]
        for ed in seq:
            if ed not in it2i:
                continue
            idx = it2i[ed]
            sim_idx, sim_sc = als.similar_items(idx, N=70)
            for j, s in zip(sim_idx, sim_sc):
                cand = int(i2it[int(j)])
                agg[cand] += float(max(0.0, s))
        als_simitem_agg[int(u)] = dict(agg)

    # 3) aggregate by nearest users in ALS space
    user_f = als.user_factors
    item_f = als.item_factors
    user_norm = np.linalg.norm(user_f, axis=1) + 1e-9

    for u in tqdm(targets['user_id'].tolist(), desc='als user-knn agg'):
        if u not in u2i:
            continue
        uid = u2i[u]
        vec = user_f[uid]
        sims = (user_f @ vec) / (user_norm * (np.linalg.norm(vec) + 1e-9))
        top_idx = np.argpartition(-sims, 120)[:120]
        top_idx = top_idx[top_idx != uid]

        agg = defaultdict(float)
        for nb in top_idx:
            w = float(max(0.0, sims[nb]))
            if w <= 0:
                continue
            nz = mat[nb].indices
            for it in nz[:140]:
                agg[int(i2it[int(it)])] += w
        als_userknn_agg[int(u)] = dict(agg)

print('HAS_IMPLICIT:', HAS_IMPLICIT, 'als_recs:', len(als_recs), 'simitem_agg:', len(als_simitem_agg), 'userknn_agg:', len(als_userknn_agg))


In [ ]:
global_list = list(zip(global_top['edition_id'].astype(int), global_top['pop_all'].astype(float)))
recent_list = list(zip(recent_top['edition_id'].astype(int), recent_top['pop_recent'].astype(float)))


def build_candidates_for_user(uid, seen_map, max_items=CANDS_PER_USER):
    uid = int(uid)
    seen = set(seen_map.get(uid, []))
    score = defaultdict(float)
    mask = defaultdict(int)

    # global priors
    for r, (ed, s) in enumerate(recent_list[:260], start=1):
        if ed in seen:
            continue
        score[ed] += 0.90 * (s / (1 + 0.01*r)); mask[ed] |= 1
    for r, (ed, s) in enumerate(global_list[:280], start=1):
        if ed in seen:
            continue
        score[ed] += 0.55 * (s / (1 + 0.01*r)); mask[ed] |= 2

    # co-vis
    hist = seen_map.get(uid, [])[-35:]
    for t, ed0 in enumerate(hist):
        for cand, s in item2top.get(int(ed0), []):
            if cand in seen:
                continue
            score[cand] += 1.30 * (1 + 0.015*t) * s; mask[cand] |= 4

    # metadata affinity
    for aid, w in ua.get(uid, [])[:16]:
        for ed, s in author2eds.get(aid, [])[:70]:
            if ed in seen:
                continue
            score[ed] += 1.20 * w * (1 + math.log1p(s)); mask[ed] |= 8
    for gid, w in ug.get(uid, [])[:16]:
        for ed, s in genre2eds.get(gid, [])[:70]:
            if ed in seen:
                continue
            score[ed] += 1.15 * w * (1 + math.log1p(s)); mask[ed] |= 16
    for lid, w in ul.get(uid, [])[:10]:
        for ed, s in lang2eds.get(lid, [])[:75]:
            if ed in seen:
                continue
            score[ed] += 0.95 * w * (1 + math.log1p(s)); mask[ed] |= 32
    for pid, w in up.get(uid, [])[:10]:
        for ed, s in pub2eds.get(pid, [])[:75]:
            if ed in seen:
                continue
            score[ed] += 0.85 * w * (1 + math.log1p(s)); mask[ed] |= 64

    # ALS direct + neighbor aggregates
    for ed, s in als_recs.get(uid, [])[:180]:
        if ed in seen:
            continue
        score[ed] += 1.25 * max(0.0, s); mask[ed] |= 128
    for ed, s in sorted(als_simitem_agg.get(uid, {}).items(), key=lambda x: -x[1])[:220]:
        if ed in seen:
            continue
        score[ed] += 0.85 * max(0.0, s); mask[ed] |= 256
    for ed, s in sorted(als_userknn_agg.get(uid, {}).items(), key=lambda x: -x[1])[:220]:
        if ed in seen:
            continue
        score[ed] += 0.85 * max(0.0, s); mask[ed] |= 512

    ranked = sorted(score.items(), key=lambda x: (-x[1], x[0]))[:max_items]
    return ranked, mask


In [ ]:
# target candidates
cand_rows = []
for uid in tqdm(targets['user_id'].tolist(), desc='target cands'):
    ranked, m = build_candidates_for_user(uid, user_pos_full, max_items=CANDS_PER_USER)
    for ed, s in ranked:
        cand_rows.append((int(uid), int(ed), float(s), int(m[ed])))

cands = pd.DataFrame(cand_rows, columns=['user_id','edition_id','cand_score','src_mask'])
print('cands shape:', cands.shape, 'avg/user:', cands.shape[0]/max(1, len(targets)))


In [ ]:
# training table from pseudo split
truth_map = valid_truth.groupby('user_id')['edition_id'].agg(set).to_dict()
train_users = list(truth_map.keys())
train_rows = []

for uid in tqdm(train_users, desc='train cands'):
    ranked, m = build_candidates_for_user(uid, user_pos_hist, max_items=CANDS_PER_USER)
    # force positives into training frame (critical anti "all targets are equal")
    forced_pos = truth_map.get(uid, set())
    added = set()

    pos_cnt = 0
    neg_cnt = 0
    for ed, s in ranked:
        y = int(ed in forced_pos)
        if y == 1 and pos_cnt < 120:
            train_rows.append((uid, ed, s, int(m[ed]), 1)); pos_cnt += 1; added.add(ed)
        elif y == 0 and neg_cnt < NEG_PER_USER:
            train_rows.append((uid, ed, s, int(m[ed]), 0)); neg_cnt += 1
        if neg_cnt >= NEG_PER_USER and pos_cnt >= min(25, len(forced_pos)):
            break

    # add missing positives explicitly
    for ed in forced_pos:
        if ed in added:
            continue
        train_rows.append((uid, int(ed), 3.0, 0, 1))

train_df = pd.DataFrame(train_rows, columns=['user_id','edition_id','cand_score','src_mask','label'])
print('train_df:', train_df.shape, 'label_dist:', train_df['label'].value_counts(dropna=False).to_dict())


In [ ]:
# Build features for cands/train
user_feat = hist_pos.groupby('user_id').agg(
    u_events=('edition_id','size'),
    u_uniq_items=('edition_id','nunique'),
).reset_index()
user_feat['u_events_log'] = np.log1p(user_feat['u_events'])

item_feat = hist_pos.groupby('edition_id').agg(
    i_events=('user_id','size'),
    i_uniq_users=('user_id','nunique'),
).reset_index()
item_feat['i_events_log'] = np.log1p(item_feat['i_events'])

user_author_w = u_author.set_index(['user_id','author_id'])['w'] if len(u_author) else pd.Series(dtype=float)
user_lang_w = u_lang.set_index(['user_id','language_id'])['w'] if len(u_lang) else pd.Series(dtype=float)
user_pub_w = u_pub.set_index(['user_id','publisher_id'])['w'] if len(u_pub) else pd.Series(dtype=float)
user_genre_w = u_genre.set_index(['user_id','genre_id'])['w'] if len(u_genre) else pd.Series(dtype=float)

def genre_aff(uid, book_id):
    gs = book_to_genres.get(int(book_id), [])
    if not gs:
        return 0.0
    return max(float(user_genre_w.get((int(uid), int(g)), 0.0)) for g in gs)

def enrich(df):
    out = df.merge(editions[['edition_id','author_id','language_id','publisher_id','book_id','publication_year','age_restriction']], on='edition_id', how='left')
    out = out.merge(item_feat, on='edition_id', how='left').merge(user_feat, on='user_id', how='left')
    out['u_author_w'] = [float(user_author_w.get((u,a), 0.0)) for u,a in out[['user_id','author_id']].itertuples(index=False)]
    out['u_lang_w'] = [float(user_lang_w.get((u,l), 0.0)) for u,l in out[['user_id','language_id']].itertuples(index=False)]
    out['u_pub_w'] = [float(user_pub_w.get((u,p), 0.0)) for u,p in out[['user_id','publisher_id']].itertuples(index=False)]
    out['u_genre_w'] = [genre_aff(u,b) for u,b in out[['user_id','book_id']].itertuples(index=False)]

    # ALS aggregate features
    out['als_direct'] = [float(dict(als_recs.get(int(u), [])).get(int(e), 0.0)) for u,e in out[['user_id','edition_id']].itertuples(index=False)]
    out['als_simitem'] = [float(als_simitem_agg.get(int(u), {}).get(int(e), 0.0)) for u,e in out[['user_id','edition_id']].itertuples(index=False)]
    out['als_userknn'] = [float(als_userknn_agg.get(int(u), {}).get(int(e), 0.0)) for u,e in out[['user_id','edition_id']].itertuples(index=False)]

    for bit, name in [(1,'src_recent_pop'),(2,'src_global_pop'),(4,'src_covisit'),(8,'src_author'),(16,'src_genre'),(32,'src_lang'),(64,'src_pub'),(128,'src_als_direct'),(256,'src_als_simitem'),(512,'src_als_userknn')]:
        out[name] = ((out['src_mask'] & bit) > 0).astype('int8')

    out['cand_score_log'] = np.log1p(np.maximum(out['cand_score'], 0.0))
    out = out.fillna(0)
    return out

train_df = enrich(train_df)
cands = enrich(cands)

feature_cols = [
    'cand_score','cand_score_log',
    'u_author_w','u_lang_w','u_pub_w','u_genre_w',
    'als_direct','als_simitem','als_userknn',
    'u_events','u_uniq_items','u_events_log',
    'i_events','i_uniq_users','i_events_log',
    'publication_year','age_restriction',
    'src_recent_pop','src_global_pop','src_covisit','src_author','src_genre','src_lang','src_pub','src_als_direct','src_als_simitem','src_als_userknn',
    'author_id','language_id','publisher_id'
]
cat_features = [feature_cols.index(c) for c in ['author_id','language_id','publisher_id']]

print('features:', len(feature_cols), 'train shape:', train_df.shape, 'inference shape:', cands.shape)


In [ ]:
# Train robustly
train_df = train_df.sort_values(['user_id']).reset_index(drop=True)
label_values = train_df['label'].astype(int)

if label_values.nunique() < 2:
    print('WARNING: only one class in train labels -> fallback to NoML ranking only')
    model = None
else:
    train_pool = Pool(
        data=train_df[feature_cols],
        label=label_values.to_numpy(),
        group_id=train_df['user_id'].to_numpy(),
        cat_features=cat_features,
    )
    model = CatBoostRanker(**CATBOOST_PARAMS)
    model.fit(train_pool)


In [ ]:
# Predict + blend
if model is not None:
    pred_pool = Pool(
        data=cands[feature_cols],
        group_id=cands['user_id'].to_numpy(),
        cat_features=cat_features,
    )
    cands['pred'] = model.predict(pred_pool)
else:
    cands['pred'] = 0.0

cands['final_score'] = (
    cands['pred']
    + 0.22 * cands['cand_score_log']
    + 0.06 * np.log1p(np.maximum(cands['als_direct'], 0.0))
    + 0.05 * np.log1p(np.maximum(cands['als_simitem'], 0.0))
    + 0.05 * np.log1p(np.maximum(cands['als_userknn'], 0.0))
)
cands = cands.sort_values(['user_id','final_score','edition_id'], ascending=[True,False,True])


In [ ]:
# Final submission
fallback_global = recent_top['edition_id'].astype(int).tolist()
rows = []
for uid, g in tqdm(cands.groupby('user_id'), total=targets['user_id'].nunique(), desc='submission'):
    uid = int(uid)
    seen = set(user_pos_full.get(uid, []))
    picked, used = [], set()

    for ed in g['edition_id'].astype(int).tolist():
        if ed in used or ed in seen:
            continue
        used.add(ed)
        picked.append(ed)
        if len(picked) == TOP_K:
            break

    if len(picked) < TOP_K:
        for ed in fallback_global:
            if ed in used or ed in seen:
                continue
            used.add(ed)
            picked.append(ed)
            if len(picked) == TOP_K:
                break

    for r, ed in enumerate(picked, start=1):
        rows.append((uid, int(ed), int(r)))

submission = pd.DataFrame(rows, columns=['user_id','edition_id','rank'])
print('submission shape:', submission.shape)
print('rows/user min/max:', submission.groupby('user_id').size().min(), submission.groupby('user_id').size().max())
print('unique ranks:', submission.groupby('user_id')['rank'].nunique().min())
print('unique editions:', submission.groupby('user_id')['edition_id'].nunique().min())
submission.to_csv('submission.csv', index=False)
submission.head(20)
